[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 困难：Flash 注意力（分块）

实现 **带在线 softmax 的分块注意力** — Flash 注意力的核心思想。

### 函数签名
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # 返回: (B, S, D) — 与标准注意力相同
```

### 核心洞察
不实例化完整的 S×S 注意力矩阵，而是分块处理：
1. 对于每个 Q 块，遍历 K/V 块
2. 使用 **在线 softmax**：追踪运行中的 `max` 和 `sum`
3. 当 max 变化时重新缩放累加器：`acc *= exp(old_max - new_max)`
4. 最后：`output = acc / row_sum`

必须与标准 softmax 注意力产生 **相同** 的结果。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ 在此实现你的代码

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    pass

In [ ]:
# 🧪 调试
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('匹配:', torch.allclose(out, ref, atol=1e-4))

In [ ]:
# ✅ 提交
from torch_judge import check
check('flash_attention')